# Demo of Moran's I source masking 

This notebook demos the source masking and block statistics tools from the `moransi_sourcemasking` package.
The global Moran's $I$ statistic is applied to measure the autocorrelation in sliding patches of the image
(typically 10x10 to 20x20 pixels). The background masking is based on the idea that patches of the image that don't have any sources in
them will have a low autocorrelation between adjacent pixels, while patches that have sources in them will have
a higher autocorrelation. 

DQ DO_NOT_USE pixels are masked in this computation. 

One can choose to identify a fixed percentage of pixels as sky (30-40% seems to work well) or set a threshold in the 
value of the $I$ statistic (0.3-0.4 seems to work well). 

This notebook illustrates using Photutils to do background estimation on the source-masked image, and
then looks at the resulting statistics of the background-subtracted image. There is a `block_average_robust` routine 
as part of the package that does some sigma clipping when block averaging. 


# Package installation instructions

The package currently resides in a development repository

Install with:
    `pip install git+hcferguson/moransi_sourcemasking.git`

or for local development:
```
    git clone https://github.com/hcferguson/moransi_sourcemasking.git
    cd moransi_sourcemasking
    pip install -e .
```
Restart this notebook's kernel after installation.

In [ ]:
# Arrays and plotting
import numpy as np
from matplotlib import pyplot as plt

# For making fake data
from photutils.datasets import make_100gaussians_image, make_noise_image, make_model_params, make_model_image, apply_poisson_noise
from photutils.psf import GaussianPSF
from photutils.aperture import EllipticalAperture

# For background subtraction
from photutils.background import Background2D, MedianBackground
from photutils.background import BkgZoomInterpolator
from astropy.stats import SigmaClip

# For source masking and superpixel analysis
from moransi_sourcemasking import read_config, make_sourcemask, block_average_robust
from astropy.stats import mad_std

# Set up logging
import logging
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")
logging.getLogger("moransi_sourcemasking").setLevel(logging.INFO)

# For this test, we will read the config file from a string, so we'll import these
import yaml
from box import Box
from pprint import pprint # For inspecting the read-in parameters

In [ ]:
# Create pure noise image & weight maps
N=2048
img_shape = (N,N)

# Optionally, have different noise and weights on the left and right halves
noise_factor = 1.5
half = int(N/2)
right = (slice(0,-1), slice(half,-1))
left = (slice(0,-1), slice(0,half))

# Create the noise image
noise = make_noise_image(img_shape,distribution='gaussian',mean=0.,stddev=1)
noise2 = make_noise_image(img_shape,distribution='gaussian',mean=0.,stddev=noise_factor)
wht = np.ones_like(noise)*1./np.var(noise)
wht2 = np.ones_like(noise)*1./np.var(noise2)

# Replace the right hand side with the higher noise
noise[right] = noise2[right]
wht[right] = wht2[right]
wht.min(),wht.max()

In [ ]:
# Display the noise
vmin, vmax = np.percentile(noise,(1,99))
plt.imshow(noise,vmin=vmin,vmax=vmax,cmap=plt.cm.gray,origin='lower');

In [ ]:
# Add sources
model = GaussianPSF()
model_shape = (100, 100)
Nsource = int(1000 * np.sqrt(2048.**2/(img_shape[0]*img_shape[1])))
print(Nsource)
source_catalog = make_model_params(img_shape, Nsource, x_name='x_0',
                           y_name='y_0', min_separation=5,
                           flux=(10,2000), x_fwhm=(1, 30),
                           y_fwhm=(1, 30), theta=(0, 90), seed=123)
source_catalog = make_model_params(img_shape, Nsource, x_name='x_0',
                           y_name='y_0', min_separation=5,
                           mag=(15,25),
                           theta=(0, 180), seed=123)
source_catalog['flux'] = 10.**((28-source_catalog['mag'])/2.5)
flux0 = 1000
beta = 0.4
source_catalog['logre'] = np.random.lognormal(mean=0,sigma=0.3,size=len(source_catalog)) * (source_catalog['flux']/flux0)**beta
a = source_catalog['x_fwhm'] = 10.**0.8*source_catalog['logre']
b = np.random.uniform(0.2,1,a.shape) * a
source_catalog['y_fwhm'] = b
source_catalog['model_shape'] = (5*a).astype(np.int32)
data = make_model_image(img_shape, model, source_catalog, model_shape=model_shape,x_name='x_0', y_name='y_0')
data = apply_poisson_noise(data)
image = data + noise

In [ ]:
# Display the image with added sources
plt.figure(figsize=(12,12))
vmin, vmax = np.percentile(image,(2,95))
plt.imshow(image,vmin=vmin,vmax=vmax,cmap=plt.cm.gray,origin="lower");

## Set up the control parameters

 # These are only the parameters that really matter for global I source masking
```
kernel_width:          3  # This kernel is a 3x3 box with a 0 in the center
patch_size:           20  # This is the patch full width in pixels
n_box_passes:          1  # Boxcar smooth the I statistic array before thresholding
opening_iterations:    2  # Remove small disconnected sources in the mask
threshold_type: "percentage" # "I value" or "percentage"
threshold_value:    35  # Threshold in percent or in I value (30-40% or I=0.3-0.4 generally work)
```

Ordinarily we would have the parameters in a yaml file. Then to read it, just have a line like:
```
config = read_config('sourcemask_parameters.yaml')
```
In this example we will put the parameters in a string, but as yaml, and read it. We could have just put them in a dictionary. 


In [ ]:
# Set up the parameters for the different tiers, running from huge sources to tiny sources
#
yaml_string = """
---
kernel_width:             3  # This kernel is a 3x3 box with a 0 in the center
patch_size:              20  # This is the patch full width in pixels
n_box_passes:             1  # Boxcar smooth the I statistic array before thresholding
opening_iterations:       2  # Remove small disconnected sources in the mask
threshold_type: "percentage" # "I value" or "percentage"
threshold_value:         35  # Threshold in percent or in I value (30-40% or I=0.3-0.4 generally work)
"""

# Read in this string into the object returned by read_config()
config = box_data = Box(**yaml.safe_load(yaml_string))
print(config.to_yaml())

In [ ]:
%%time
# Make the source masks one by one; this takes about 25 seconds for this demo
bad_mask = np.zeros(image.shape,dtype='bool')
source_mask = make_sourcemask(image,config,bad_mask=bad_mask,weight=wht)

In [ ]:
# Plot the image and source mask
subset = image[:,0:100]
vmin, vmax = np.percentile(subset,(1,80))
fig, ax = plt.subplots(1,3,figsize=(24,8),layout='constrained',sharex=True,sharey=True)
ax[0].imshow(image,vmin=vmin,vmax=vmax,cmap=plt.cm.gray,origin='lower')
ax[1].imshow(image*source_mask,vmin=vmin,vmax=vmax,cmap=plt.cm.gray,origin='lower')
ax[2].imshow(source_mask,vmin=0,vmax=1,cmap=plt.cm.gray,origin='lower')
ax[0].set_title("Original image")
ax[1].set_title("Masked")
ax[2].set_title("Source mask");

# Add a spatially varying background

In [ ]:
# Add a nonlinear gradient in x and y
width,height = img_shape
y,x = np.mgrid[0:width, 0:height]

# Number of random backgrounds to add
N = 500
amplitude_max = 50./N

# Put in an initial sinusoidal background
cy,cx = 400,400
period = 300
r = np.sqrt((x-cx)**2+(y-cy)**2)
amplitude = amplitude_max
flux1 = amplitude*np.sin((r/period)*2*np.pi)*np.exp(-2*r/width)
flux = flux1

# Add a bunch of intersecting backgrounds
cx0,cx1 = -img_shape[0]/3, 1.333*img_shape[0]
cy0,cy1 = -img_shape[0]/3, 1.333*img_shape[0]
cys = np.random.uniform(cy0,cy1,N)
cxs = np.random.uniform(cx0,cx1,N)
periods = np.random.uniform(300,4000,N)
amplitudes = np.random.uniform(amplitude_max/10.,amplitude_max,N)
for cx, cy, period, amplitude in zip(cxs,cys,periods,amplitudes):
    r = np.sqrt((x-cx)**2+(y-cy)**2)
    flux += amplitude * np.sin((r/period)*2*np.pi)

new_image = image + flux

In [ ]:
# Compare with and without this dirty sky background
vmin, vmax = np.percentile(image,(2,90))
fig, ax = plt.subplots(1,2,figsize=(16,8), layout='constrained')
ax[0].imshow(image,vmin=vmin,vmax=vmax,cmap=plt.cm.gray,origin='lower')
ax[1].imshow(new_image,vmin=vmin,vmax=vmax,cmap=plt.cm.gray,origin='lower');

In [ ]:
# Make a new source mask. 
bad_mask = np.zeros(new_image.shape,dtype='bool')
source_mask = make_sourcemask(new_image,config,bad_mask=bad_mask,weight=wht)

In [ ]:
# Display the new mask
fig, ax = plt.subplots(1,3,figsize=(24,8), layout='constrained')
ax[0].imshow(new_image,vmin=vmin,vmax=vmax,cmap=plt.cm.gray,origin='lower')
ax[1].imshow(new_image*source_mask,vmin=vmin,vmax=vmax,cmap=plt.cm.gray,origin='lower')
ax[2].imshow(source_mask,vmin=0,vmax=1,cmap=plt.cm.gray,origin='lower');

In [ ]:
# A function to fit the background using photutils Background2D
def fit_background(image,error,inputmask,box_size=8,
                   sigma = 3.,
                   bkg_estimator = MedianBackground(),
                   filter_size = (11,11)
                   ):
    sigma_clip = SigmaClip(sigma=sigma)
    mask = np.isnan(image) | inputmask
    coverage_mask = np.logical_or(np.isnan(error), error == 0)

    n_masked = np.count_nonzero(mask)
    n_coverage_masked = np.count_nonzero(coverage_mask)
    print(f"Fraction of pixels masked: {n_masked/image.size:.3f} ({n_masked} pixels)")
    print(f"Pixels removed by coverage mask: {n_coverage_masked/image.size:.3f} ({n_coverage_masked} pixels)")

    result = Background2D (
        image,
        box_size,
        filter_size=filter_size,
        mask=mask,
        coverage_mask=coverage_mask,
        sigma_clip=sigma_clip,
        bkg_estimator=bkg_estimator,
    )

    return result

In [ ]:
# Fit the sky background, masking out the sources
bkg = fit_background(new_image,1./np.sqrt(wht),~source_mask)
unmaskedbkg = fit_background(new_image,1./np.sqrt(wht),np.zeros_like(source_mask))

In [ ]:
# Display before and after
vmin, vmax = np.percentile(new_image,(2,90))
fig, ax = plt.subplots(1,2,figsize=(16,8), layout='constrained')
ax[0].imshow(new_image,vmin=vmin,vmax=vmax,cmap=plt.cm.gray,origin='lower')
ax[1].imshow(new_image-bkg.background,vmin=vmin,vmax=vmax,cmap=plt.cm.gray,origin='lower');

## Block-Averaged super-pixel analysis
A fairly intuitive way to assess the flatness of the sky is to look at the rms as a function of scale. If we have an error array, we can compare the measured sky RMS from the source-free sky pixels to the predicted standard deviation from the error array, as a function of the super-pixel size.  

Note:

 - If the sky has roughly constant `err` and is smooth, we expect ratios of sky RMS `err` ~1, independent of scale
 - If the sky is not smooth this ratio will probably be $>1$ and a function of scale
 - If the `err` array itself varies across the image, values of the ratio could be $<1$

In [ ]:
# Do the background subtraction
bkgsub = new_image - bkg.background
unmasked_bkgsub = new_image - unmaskedbkg.background

# Create an err array from the weight map
err = 1./np.sqrt(wht)

In [ ]:
# Routine to return block-averaged statistics for a range of block sizes
def block_averaged_stats(image,err,source_mask,block_sizes=np.arange(3,30)):
    # Set up empty arrays to receive the results
    rms_skys = np.zeros(block_sizes.shape,dtype=np.float64)
    ratios = np.zeros_like(rms_skys)

    # Compute the statistics
    for i,blk_size in enumerate(block_sizes):
        # skip superpixels with too few unmasked pixels
        min_good = int(blk_size*blk_size/2) 
        # Measure the block averages with some sigma clipping
        blk_image, blk_err, blk_mask = block_average_robust(image, err, source_mask, block_size=blk_size, min_good=min_good)
        # Robust statistics of the ratio measured_sky_RMS/predicted_from_err
        rms_skys[i] = mad_std(blk_image[blk_mask])
        ratios[i] = mad_std(blk_image[blk_mask]) / blk_err[blk_mask].mean()
    return rms_skys, ratios, block_sizes

In [ ]:
# Compute both for the original image and the background-subtracted image with the ugly background
rms_skys_clean_image, ratios_clean_image, block_sizes = block_averaged_stats(image,err,source_mask)
rms_skys_dirty_image, ratios_dirty_image, block_sizes = block_averaged_stats(new_image,err,source_mask)
rms_skys_bkgsub, ratios_bkgsub, block_sizes = block_averaged_stats(bkgsub,err,source_mask)
rms_skys_unmasked_bkgsub, ratios_unmasked_bkgsub, block_sizes = block_averaged_stats(unmasked_bkgsub,err,source_mask)

In [ ]:
# Plot this ratio vs. super-pixel scale for the clean image
fig, ax = plt.subplots(1,2,figsize=(16,8), layout='constrained')
ax[0].plot(block_sizes,rms_skys_clean_image*block_sizes, label='clean image')
ax[0].plot(block_sizes,rms_skys_bkgsub*block_sizes, label="dirty sky subtracted")
ax[0].plot(block_sizes,rms_skys_unmasked_bkgsub*block_sizes, label="unmasked sky subtracted")
ax[0].set_xlabel("super-pixel size (pixels)",fontsize=16)
ax[0].set_ylabel("Measured sky RMS",fontsize=16)
ax[0].legend()

ax[1].plot(block_sizes,ratios_clean_image, label = 'clean_image')
ax[1].plot(block_sizes,ratios_bkgsub, label="dirty sky subtracted")
ax[1].plot(block_sizes,ratios_unmasked_bkgsub, label="unmasked sky subtracted")
ax[1].set_xlabel("super-pixel size (pixels)",fontsize=16)
ax[1].set_ylabel("ratio sky RMS(measured) / mean(err)",fontsize=16)
ax[1].legend();

If source masking and background subtraction had been perfect, both plots would be flat as a function of scale. If the error array were perfect, the right-hand plot would be flat at 1.0. 

The plots show that background subtraction is imperfect, and give and indication of how imperfect as a function of scale. Scales of 10 superpixels might be typical of faint galaxies in Roman image, so in the rms sky fluctuations left after background subtraction are quite close to the ideal. 

To get a sense for the improvement due to the background subtraction, let's include the "dirty" image in the same plot.

In [ ]:
# Include the dirty image in the plots
# Plot this ratio vs. super-pixel scale for the clean image
fig, ax = plt.subplots(1,2,figsize=(16,8), layout='constrained')
ax[0].plot(block_sizes,rms_skys_clean_image*block_sizes, label='clean image')
ax[0].plot(block_sizes,rms_skys_dirty_image*block_sizes, label='dirty image')
ax[0].plot(block_sizes,rms_skys_bkgsub*block_sizes, label="dirty sky subtracted")
ax[0].set_xlabel("super-pixel size (pixels)",fontsize=16)
ax[0].set_ylabel("Measured sky RMS",fontsize=16)
ax[0].legend()

ax[1].plot(block_sizes,ratios_clean_image, label = 'clean_image')
ax[1].plot(block_sizes,ratios_dirty_image, label = 'dirty_image')
ax[1].plot(block_sizes,ratios_bkgsub, label="dirty sky subtracted")
ax[1].set_xlabel("super-pixel size (pixels)",fontsize=16)
ax[1].set_ylabel("ratio sky RMS(measured) / mean(err)",fontsize=16)
ax[1].legend();

## Compare left and right sides

If the noise was set to be different in the left and right halves, the ratio of the measured sky RMS to the "predicted" RMS from the err array won't be quite 1. It should be very close to 1 if you restrict to a region of constant sky uncertainty.

In [ ]:
# Compute both for the original image and the background-subtracted image with the ugly background
slices = {'left':left, 'right':right}
images = {'clean':image, 'dirty':new_image, 'bkgsub':bkgsub, 'unmasked_bkgsub':unmasked_bkgsub}
rms_skys = {}
ratios = {}

for imkey in images.keys():
    for lr in slices.keys():
        iyx = slices[lr]
        rms_skys[(imkey,lr)], ratios[(imkey,lr)], block_sizes = block_averaged_stats(
                       images[imkey][iyx],err[iyx],source_mask[iyx])

In [ ]:
# Plot this ratio vs. super-pixel scale for the clean image
fig, ax = plt.subplots(2,2,figsize=(16,16), layout='constrained')
ax[0,0].plot(block_sizes,rms_skys[('clean','left')]*block_sizes, label='clean image')
ax[0,0].plot(block_sizes,rms_skys[('bkgsub','left')]*block_sizes, label="dirty sky subtracted")
ax[0,0].plot(block_sizes,rms_skys[('unmasked_bkgsub','left')]*block_sizes, label="unmasked sky subtracted")
ax[0,0].set_xlabel("super-pixel size (pixels)",fontsize=16)
ax[0,0].set_ylabel("Measured sky RMS",fontsize=16)
ax[0,0].legend()
ax[0,0].set_title('Left')

ax[0,1].plot(block_sizes,ratios[('clean','left')], label = 'clean_image')
ax[0,1].plot(block_sizes,ratios[('bkgsub','left')], label="dirty sky subtracted")
ax[0,1].plot(block_sizes,ratios[('unmasked_bkgsub','left')], label="unmasked sky subtracted")
ax[0,1].set_xlabel("super-pixel size (pixels)",fontsize=16)
ax[0,1].set_ylabel("ratio sky RMS(measured) / mean(err)",fontsize=16)
ax[0,1].legend()
ax[0,1].set_title('Left')

ax[1,0].plot(block_sizes,rms_skys[('clean','right')]*block_sizes, label='clean image')
ax[1,0].plot(block_sizes,rms_skys[('bkgsub','right')]*block_sizes, label="dirty sky subtracted")
ax[1,0].plot(block_sizes,rms_skys[('unmasked_bkgsub','right')]*block_sizes, label="unmasked sky subtracted")
ax[1,0].set_xlabel("super-pixel size (pixels)",fontsize=16)
ax[1,0].set_ylabel("Measured sky RMS",fontsize=16)
ax[1,0].legend()
ax[1,0].set_title('Right')

ax[1,1].plot(block_sizes,ratios[('clean','right')], label = 'clean_image')
ax[1,1].plot(block_sizes,ratios[('bkgsub','right')], label="dirty sky subtracted")
ax[1,1].plot(block_sizes,ratios[('unmasked_bkgsub','right')], label="unmasked sky subtracted")
ax[1,1].set_xlabel("super-pixel size (pixels)",fontsize=16)
ax[1,1].set_ylabel("ratio sky RMS(measured) / mean(err)",fontsize=16)
ax[1,1].legend()
ax[1,1].set_title('Right');